# Synthetic two-speaker conversations for caramel **pretraining**

Same pipeline as [synthetic_data.ipynb](synthetic_data.ipynb), but for the general conversation stage instead of the JOSIE assistant:

| | `synthetic_data.ipynb` (assistant fine-tune) | this notebook (pretraining) |
|---|---|---|
| speaker A | JOSIE, always the voice *Zoey* | random voice per conversation |
| speaker B | random user voice | random voice, never the same as A |
| persona | "You are JOSIE, a voice assistant" | none: two people talking |
| train with | `train.py --no_swap` (learn only JOSIE's side) | `train.py` default (both sides, speakers swapped) |

Pretraining teaches the model to *converse* (turn-taking, barge-ins, backchannels, holds) from both sides and in many voices.
The assistant stage afterwards fixes who it is: one voice, one side, yielding when interrupted.

Output goes to `HF_REPO` in chunks, exactly like the assistant notebook (see there for resume and layout).


In [ ]:
!uv pip install openai datasets ptts mlx-vlm tqdm

In [ ]:
import sys
from pathlib import Path
from openai import OpenAI
from synthetic.generator import SyntheticConfig, SyntheticConversationGenerator
from synthetic.phonon_speech import PhononSpeech, PhononVoice, phonon_voices
from synthetic.teacher import ChatTeacher, MLXTeacher, load_prompt_corpus
from synthetic.hub import generate_to_hub

In [ ]:
OUTPUT_DIR = Path("data/synthetic_pretrain")
HF_REPO = "Goekdeniz-Guelmez/caramel-synthetic-pretrain"   # dataset repo (`hf auth login` first); created private if missing
CHUNK_SIZE = 8     # conversations per upload; at most one chunk of audio is ever on disk
WORKERS = 8        # conversations generated in parallel (teacher calls overlap; TTS/alignment share the GPU)
MAX_SAMPLES = None # None = whole prompt corpus
SYSTEM_PROMPT = ("You are a person in a relaxed spoken conversation with someone you know. Answer naturally and briefly, "
                 "share your own opinions and experiences, ask back sometimes. Spoken aloud: no markdown, code, lists or math.")

TEACHER = "ollama"   # "mlx": local MLX-VLM model, "ollama": OpenAI-compatible endpoint

if TEACHER == "mlx":
    teacher = MLXTeacher("mlx-community/gemma-4-e4b-it-4bit", max_tokens=512)
elif TEACHER == "ollama":
    # Fail visibly after a stalled request instead of waiting through SDK retries.
    client = OpenAI(
        base_url="http://localhost:11434/v1", api_key="ollama",
        timeout=120.0, max_retries=0,
    )
    teacher = ChatTeacher(client, assistant_model="deepseek-v4.1-flash:cloud", followup_model="deepseek-v4.1-flash:cloud")
else:
    raise ValueError(f"Unknown TEACHER: {TEACHER}")

# phonon-40m-multilingual is gated (CC BY-NC 4.0): accept the terms on Hugging Face, then `hf auth login`.
# Voices: PhononVoice(name="Zoey") from `python -m ptts --list-voices`, PhononVoice(file="voice.safetensors"),
# or PhononVoice(wav="reference.wav") if the checkpoint supports cloning.
# User voice: one is picked at random per conversation from this pool. Select by name or filter, e.g.
#   phonon_voices("Zoey", "Hugh", "Maeve")  |  phonon_voices(accent="uk")  |  phonon_voices(gender="male")
VOICES = phonon_voices()   # all US + UK catalog voices; each conversation draws two different ones
speech = PhononSpeech(
    OUTPUT_DIR / "_speech_cache",
    model="hf://gradium/phonon-40m-multilingual",   # or "hf://gradium/phonon-31m-en" (Apache 2.0)
    language="en",
    user_voice=VOICES,
    assistant_voice=VOICES,          # a list = random speaker-A voice per conversation
    min_alignment_match=0.8,
)
config = SyntheticConfig(
    max_turns=8, barge_in_probability=0.25, assistant_stop_ms=750, system_prompt=SYSTEM_PROMPT, seed=42,
    wait_probability=0.10,         # "can you wait a minute?" -> silent while the user is busy, then resume
    listen_probability=0.10,       # "listen until I finish my presentation" -> silent, then feedback
    max_wait_seconds=60,           # longest timed wait ("a minute")
    check_in_probability=0.5,      # "the minute is up, can I continue?" / "not yet, I'll tell you"
    backchannel_probability=0.2,   # "mhm", "right", "yeah" in the pauses of long user turns
)


In [ ]:
prompts, source = load_prompt_corpus("mlx-community/ultrafeedback-prompts-flat-rlhf")
generator = SyntheticConversationGenerator(teacher, speech, OUTPUT_DIR, config, source_metadata=source)
# Capture all thread stacks if TTS, alignment, or the teacher stops progressing.
# The log also works in kernels whose stderr does not expose a file descriptor.
import faulthandler
stall_log = OUTPUT_DIR / "generation_stall.log"
print(f"Stall diagnostics (after 120 seconds): {stall_log.resolve()}", flush=True)
with stall_log.open("w") as diagnostic_file:
    faulthandler.dump_traceback_later(120, repeat=True, file=diagnostic_file)
    try:
        meta = generate_to_hub(generator, prompts, HF_REPO, chunk_size=CHUNK_SIZE, max_samples=MAX_SAMPLES,
                               speech_cache=OUTPUT_DIR / "_speech_cache", workers=WORKERS)
    finally:
        faulthandler.cancel_dump_traceback_later()
print(f"{len(meta['chunks'])} chunks, {meta['frames']['train'] / 12.5 / 3600:.2f} h train, {meta['frames']['validation'] / 12.5 / 3600:.2f} h validation on {HF_REPO}")

## Train on it

```bash
# 1. pretrain / continue pretraining on both sides (mix with real conversations if you have them)
python train.py --data hf://Goekdeniz-Guelmez/caramel-synthetic-pretrain --dev hf://Goekdeniz-Guelmez/caramel-synthetic-pretrain \
  --init ./caramel-1 --epochs 2 --lr 1e-4 --eval_every 50 --save_best --out runs/pretrain-synth
# 2. assistant fine-tune: only JOSIE's side
python train.py --data hf://Goekdeniz-Guelmez/josie-live-synthetic --dev hf://Goekdeniz-Guelmez/josie-live-synthetic \
  --init runs/pretrain-synth/best --no_swap --epochs 3 --lr 1e-4 --eval_every 50 --save_best --out runs/josie
```
